# Gene ↔ Pathway Relation-Wise Merge

Merges Gene–Pathway triples from Monarch, TARKG, iBKH, and Harmonizome; resolves
missing gene head names via NCBI synonyms; normalises ID-type labels; deduplicates by
`(head, relation, tail)`; and saves the result.

## 0. Configuration

In [1]:
import os
import pandas as pd
import numpy as np

BASE_DIR  = '/storage/Arushi/090526_EvoAge/kg_formation/'
PROC_DIR  = BASE_DIR + 'processed_data/'
DB_DIR     = BASE_DIR + 'data_collection/databases_for_mapping/'

OUT_PATH  = BASE_DIR + 'processed_data_relation_wise_merge/generalised/GENE_PATHWAY/ALL_GENE_PATHWAY.csv'

REQUIRED_COLS = [
    'head', 'relation', 'tail',
    'head_type', 'relation_type', 'tail_type',
    'kg_source',
    'head_id_is', 'tail_id_is',
    'head_detail_name', 'tail_detail_name', 'kg_type', 'species'
]

## 1. Build Gene Lookup Dictionaries — NCBI and ENSEMBL

In [2]:
ENS_2NCBI = pd.read_csv(DB_DIR + 'ENSEMBL/ENSEMBLE_ID_2_NCBI_Gene_SYM.csv')
ENS_2NCBI = ENS_2NCBI[~ENS_2NCBI['name'].isna()]
NCBI_2ENS__dict = dict(zip(ENS_2NCBI['name'], ENS_2NCBI['initial_alias']))
del ENS_2NCBI

NCBI_ALL_GENE = pd.read_csv(DB_DIR + 'ncbi/Homo_sapiens.gene_info', sep='\t')
NCBI_ALL_GENE['ENSEMBLE_ID'] = NCBI_ALL_GENE['Symbol'].map(NCBI_2ENS__dict)
NCBI_ALL_GENEname_dict   = dict(zip(NCBI_ALL_GENE['GeneID'], NCBI_ALL_GENE['description']))
NCBI_ALL_GENEIDname_dict = dict(zip(NCBI_ALL_GENE['GeneID'], NCBI_ALL_GENE['Symbol']))
NCBI_ALL_Symb_Desc_dict  = dict(zip(NCBI_ALL_GENE['Symbol'], NCBI_ALL_GENE['description']))

# Exploded synonym → canonical Symbol dict
NCBI_ALL_GENE_Syn_Dict = dict(zip(NCBI_ALL_GENE['Synonyms'], NCBI_ALL_GENE['Symbol']))
exploded_dict_NCBI_ALL_GENE_Syn_Dict = {}
for k, v in NCBI_ALL_GENE_Syn_Dict.items():
    for alias in k.split('|'):
        exploded_dict_NCBI_ALL_GENE_Syn_Dict[alias.strip()] = v

print(f"NCBI gene table: {len(NCBI_ALL_GENE):,} rows")
print(f"Synonym dict:    {len(exploded_dict_NCBI_ALL_GENE_Syn_Dict):,} entries")

NCBI gene table: 193,505 rows
Synonym dict:    69,564 entries


In [22]:
## 1c Kegg pathway

kegg_pathway = pd.read_csv(DB_DIR + 'kegg/kegg_pathways.txt', sep='\t', header=None)
kegg_pathway[0] = 'KEGG:' + kegg_pathway[0]
kegg_pathway_dict = dict(zip(kegg_pathway[0], kegg_pathway[1]))
# kegg_pathway

## 2. Load KG Sources

### 2a. Monarch

In [4]:
Monarch_Gene_Pathway = pd.read_csv(PROC_DIR + 'Monarch/Human/Human_Gene_Pathway_human_MONARCH.csv')
Monarch_Gene_Pathway.columns = Monarch_Gene_Pathway.columns.str.lower()
Monarch_Gene_Pathway.rename(columns={'tail_name': 'tail_detail_name'}, inplace=True)
Monarch_Gene_Pathway['kg_source'] = 'MonarchKG'
print(f"Monarch:     {len(Monarch_Gene_Pathway):,} rows")
Monarch_Gene_Pathway['head_id_is'] = 'NCBI_ID'

Monarch_Gene_Pathway['kg_type'] = 'Generalised'

Monarch_Gene_Pathway['species'] = 'Homo species'

Monarch_Gene_Pathway.head(2)

Monarch:     39,128 rows


,head,relation,tail,head_type,relation_type,tail_type,relation_source,kg_source,head_detail_name,head_taxon_name,tail_taxon_name,head_id_is,tail_id_is,head_name,tail_detail_name,head_orignal,species_species,kg_type,species
0,ASMT,Gene_Pathway,R-HSA-209931,Gene,participates_in,Pathway,infores:reactome,MonarchKG,acetylserotonin O-methyltransferase,Homo sapiens,Homo sapiens,NCBI_ID,Reactome,ASMT,Serotonin and melatonin biosynthesis,HGNC:750,Homo sapiens_Homo sapiens,Generalised,Homo species
1,KLK14,Gene_Pathway,R-HSA-6809371,Gene,participates_in,Pathway,infores:reactome,MonarchKG,kallikrein related peptidase 14,Homo sapiens,Homo sapiens,NCBI_ID,Reactome,KLK14,Formation of the cornified envelope,HGNC:6362,Homo sapiens_Homo sapiens,Generalised,Homo species


### 2b. TARKG

In [7]:
TARKG_Gene_Pathway = pd.read_csv(PROC_DIR + 'TARKG/Gene_Pathway_TARKG.csv')
TARKG_Gene_Pathway.columns = TARKG_Gene_Pathway.columns.str.lower()
print(f"TARKG:       {len(TARKG_Gene_Pathway):,} rows")

TARKG_Gene_Pathway['kg_type'] = 'Generalised'
TARKG_Gene_Pathway['species'] = 'Homo species'

TARKG_Gene_Pathway.head(2)

TARKG:       104,588 rows


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_detail_name,tail_detail_name,head_id_is,tail_id_is,kg_index,kg,change,kg_type,species
0,AHCTF1,Gene_Pathway,R-HSA-1640170,Gene,GENE_PATHWAY,Pathway,TARKG,AT-hook containing transcription factor 1,Cell Cycle,NCBI_ID,Reactome_ID,OpenBioLink-1613543,OpenBioLink,0,Generalised,Homo species
1,NUP62,Gene_Pathway,R-HSA-1640170,Gene,GENE_PATHWAY,Pathway,TARKG,nucleoporin 62,Cell Cycle,NCBI_ID,Reactome_ID,OpenBioLink-1528822,OpenBioLink,0,Generalised,Homo species


### 2c. iBKH

In [23]:
iBKH_Gene_Pathway = pd.read_csv(PROC_DIR + 'iBKH/Gene_Pathway_ibkh.csv')
iBKH_Gene_Pathway['Tail_detail_name'] = iBKH_Gene_Pathway['Tail'].map(kegg_pathway_dict)
iBKH_Gene_Pathway.columns = iBKH_Gene_Pathway.columns.str.lower()

print(f"iBKH:        {len(iBKH_Gene_Pathway):,} rows")
iBKH_Gene_Pathway['kg_type'] = 'Generalised'
iBKH_Gene_Pathway['species'] = 'Homo species'

iBKH_Gene_Pathway.head(2)

iBKH:        150,367 rows


,head,relation,tail,head_type,tail_type,kg_source,head_detail_name,tail_detail_name,head_id_is,tail_id_is,tail_detail_name.1,kg_type,species
0,A1BG,Gene_Pathway,R-HSA-109582,Gene,Pathway,iBKH,alpha-1-B glycoprotein,NaN,NCBI_ID,Reactome_ID,Hemostasis,Generalised,Homo species
1,A1BG,Gene_Pathway,R-HSA-114608,Gene,Pathway,iBKH,alpha-1-B glycoprotein,NaN,NCBI_ID,Reactome_ID,Platelet degranulation,Generalised,Homo species


### 2d. Harmonizome

In [10]:
Harmonizome_Gene_Pathway = pd.read_csv(PROC_DIR + 'harmonizome/Gene_Pathway_Harmonizome.csv')
Harmonizome_Gene_Pathway.columns = Harmonizome_Gene_Pathway.columns.str.lower()
print(f"Harmonizome: {len(Harmonizome_Gene_Pathway):,} rows")
Harmonizome_Gene_Pathway['kg_type'] = 'Generalised'
Harmonizome_Gene_Pathway['species'] = 'Homo species'
Harmonizome_Gene_Pathway.head(2)

Harmonizome: 62,384 rows


,head,relation,tail,head_type,tail_type,source,kg_source,head_detail_name,tail_detail_name,head_id_is,tail_id_is,kg_type,species
0,RAE1,Gene_Pathway,R-HSA-71387,Gene,Pathway,Reactome,Harmonizome,ribonucleic acid export 1,Metabolism of carbohydrates,NCBI_ID,Reactome,Generalised,Homo species
1,TPR,Gene_Pathway,R-HSA-71387,Gene,Pathway,Reactome,Harmonizome,"translocated promoter region, nuclear basket p...",Metabolism of carbohydrates,NCBI_ID,Reactome,Generalised,Homo species


### 2d. Hetionet

In [12]:
hetionet_Gene_Pathway = pd.read_csv(PROC_DIR + 'Hetionet/Gene_Pathway_Hetionet.csv')
hetionet_Gene_Pathway.columns = hetionet_Gene_Pathway.columns.str.lower()
print(f"hetionet: {len(hetionet_Gene_Pathway):,} rows")
hetionet_Gene_Pathway['kg_type'] = 'Generalised'
hetionet_Gene_Pathway['species'] = 'Homo species'
hetionet_Gene_Pathway.head(2)

hetionet: 84,349 rows


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_name,head_detail_name,kg_type,species
0,BUB1B,Gene_Pathway,PC7_6941,Gene,Gene–participates–Pathway,Pathway,Hetionet,NCBI_ID,Reactome,701,BUB1 mitotic checkpoint serine/threonine kinase B,Generalised,Homo species
1,ACE,Gene_Pathway,PC7_5330,Gene,Gene–participates–Pathway,Pathway,Hetionet,NCBI_ID,Reactome,1636,angiotensin I converting enzyme,Generalised,Homo species


## 3. Check and Fix Duplicate Columns

In [24]:
SOURCE_DFS = [
    ('Monarch_Gene_Pathway',     Monarch_Gene_Pathway),
    ('TARKG_Gene_Pathway',       TARKG_Gene_Pathway),
    ('iBKH_Gene_Pathway',        iBKH_Gene_Pathway),
    ('Harmonizome_Gene_Pathway', Harmonizome_Gene_Pathway),
    ('hetionet_Gene_Pathway',  hetionet_Gene_Pathway)
]

for name, df in SOURCE_DFS:
    dupes = df.columns[df.columns.duplicated(keep=False)].unique().tolist()
    if dupes:
        print(f"\n[{name}] duplicate columns: {dupes}")
        for col in dupes:
            for i, (_, s) in enumerate(df.loc[:, df.columns == col].items()):
                print(f"  '{col}' occurrence {i} → sample: {s.dropna().head(3).tolist()}")
    else:
        print(f"[{name}] ✓ no duplicates")

[Monarch_Gene_Pathway] ✓ no duplicates
[TARKG_Gene_Pathway] ✓ no duplicates
[iBKH_Gene_Pathway] ✓ no duplicates
[Harmonizome_Gene_Pathway] ✓ no duplicates
[hetionet_Gene_Pathway] ✓ no duplicates


## 4. Align Schemas and Concatenate

In [25]:
CONCAT_COLS = [
    'head', 'relation', 'tail',
    'head_type', 'relation_type', 'tail_type',
    'kg_source', 'head_id_is', 'tail_id_is',
    'head_detail_name', 'tail_detail_name', 'kg_type', 'species'
]

df_list = []
for name, df in SOURCE_DFS:
    tmp = df.loc[:, ~df.columns.duplicated()].copy()
    for col in CONCAT_COLS:
        if col not in tmp.columns:
            tmp[col] = pd.NA
    df_list.append(tmp[CONCAT_COLS])

final_df = pd.concat(df_list, ignore_index=True)
print(f"Combined: {len(final_df):,} rows")
final_df.head(2)

Combined: 440,816 rows


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name,kg_type,species
0,ASMT,Gene_Pathway,R-HSA-209931,Gene,participates_in,Pathway,MonarchKG,NCBI_ID,Reactome,acetylserotonin O-methyltransferase,Serotonin and melatonin biosynthesis,Generalised,Homo species
1,KLK14,Gene_Pathway,R-HSA-6809371,Gene,participates_in,Pathway,MonarchKG,NCBI_ID,Reactome,kallikrein related peptidase 14,Formation of the cornified envelope,Generalised,Homo species


## 5. Standardise Fixed-Value Columns

Normalise ID-type labels: gene heads → `NCBI_ID`, pathway tails → `Reactome_ID`.

In [26]:
final_df['tail_type'] = 'Pathway'

# head_id_is: HGNC / Reactome_ID artefacts → NCBI_ID
final_df['head_id_is'] = final_df['head_id_is'].str.replace('HGNC', 'NCBI_ID', regex=False)
final_df['head_id_is'] = final_df['head_id_is'].str.replace('Reactome_ID', 'NCBI_ID', regex=False)

# tail_id_is: NCBI_ID artefact → Reactome_ID; any 'React*' prefix → Reactome_ID
final_df['tail_id_is'] = final_df['tail_id_is'].str.replace('NCBI_ID', 'Reactome_ID', regex=False)
final_df['tail_id_is'] = final_df['tail_id_is'].apply(
    lambda x: 'Reactome_ID' if isinstance(x, str) and x.startswith('React') else x
)

print("Unique relation:",   set(final_df['relation']))
print("Unique tail_type:",  set(final_df['tail_type']))
print("Unique head_id_is:", set(final_df['head_id_is']))
print("Unique tail_id_is:", set(final_df['tail_id_is']))
print("Unique kg_source:",  set(final_df['kg_source']))

Unique relation: {'Gene_Pathway'}
Unique tail_type: {'Pathway'}
Unique head_id_is: {'NCBI_ID'}
Unique tail_id_is: {'Reactome_ID', nan}
Unique kg_source: {'iBKH', 'Hetionet', 'MonarchKG', 'TARKG', 'Harmonizome'}


## 6. Deduplicate

In [27]:
GROUP_COLS = ['head', 'relation', 'tail']

def merge_sources(x):
    return '::'.join(sorted(set(x.dropna())))

final_df_dedup = final_df.groupby(GROUP_COLS, as_index=False).agg({
    'head_type':        'first',
    'relation_type':    'first',
    'tail_type':        'first',
    'kg_source':         merge_sources,
    'head_id_is':       'first',
    'tail_id_is':       'first',
    'head_detail_name': 'first',
    'tail_detail_name': 'first',
    'kg_type': merge_sources,
    'species': 'first',
})
print(f"After deduplication: {len(final_df_dedup):,} rows")

After deduplication: 265,685 rows


## 7. Drop Rows with Missing Tail Names and Repair Head Names

1. Drop rows with no `tail_detail_name` (e.g. deprecated Reactome IDs).
2. For rows missing `head_detail_name`: strip `-`, map synonyms → canonical symbol, then NCBI Symbol → description.

In [28]:
# Step 1 – drop rows with no tail name (deprecated/split Reactome IDs)
before = len(final_df_dedup)
final_df_dedup = final_df_dedup[~final_df_dedup['tail_detail_name'].isna()].reset_index(drop=True)
print(f"Dropped {before - len(final_df_dedup):,} rows with missing tail_detail_name")

# Step 2 – repair missing head_detail_name
mask = final_df_dedup['head_detail_name'].isna()
print(f"Rows needing head_detail_name repair: {mask.sum():,}")

# Clean symbol (remove '-') then map synonyms → canonical symbol
final_df_dedup.loc[mask, 'head'] = final_df_dedup.loc[mask, 'head'].str.replace('-', '', regex=False)
final_df_dedup.loc[mask, 'head'] = (
    final_df_dedup.loc[mask, 'head']
    .map(exploded_dict_NCBI_ALL_GENE_Syn_Dict)
    .fillna(final_df_dedup.loc[mask, 'head'])
)

# NCBI Symbol → description
mask2 = final_df_dedup['head_detail_name'].isna()
final_df_dedup.loc[mask2, 'head_detail_name'] = final_df_dedup.loc[mask2, 'head'].map(NCBI_ALL_Symb_Desc_dict)

print(f"Still missing head_detail_name: {final_df_dedup['head_detail_name'].isna().sum():,}")

Dropped 140,727 rows with missing tail_detail_name
Rows needing head_detail_name repair: 32
Still missing head_detail_name: 0


## 8. Add Schema Columns and Enforce Column Order

In [29]:
final_df_dedup['head_type'] = 'Gene'

final_df_dedup = final_df_dedup[REQUIRED_COLS]
print(f"Final shape: {final_df_dedup.shape}")
final_df_dedup.head()

Final shape: (124958, 13)


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name,kg_type,species
0,A1BG,Gene_Pathway,R-HSA-109582,Gene,GENE_PATHWAY,Pathway,TARKG::iBKH,NCBI_ID,Reactome_ID,alpha-1-B glycoprotein,Hemostasis,Generalised,Homo species
1,A1BG,Gene_Pathway,R-HSA-114608,Gene,participates_in,Pathway,MonarchKG::TARKG::iBKH,NCBI_ID,Reactome_ID,alpha-1-B glycoprotein,Platelet degranulation,Generalised,Homo species
2,A1BG,Gene_Pathway,R-HSA-168249,Gene,GENE_PATHWAY,Pathway,TARKG::iBKH,NCBI_ID,Reactome_ID,alpha-1-B glycoprotein,Innate Immune System,Generalised,Homo species
3,A1BG,Gene_Pathway,R-HSA-168256,Gene,GENE_PATHWAY,Pathway,TARKG::iBKH,NCBI_ID,Reactome_ID,alpha-1-B glycoprotein,Immune System,Generalised,Homo species
4,A1BG,Gene_Pathway,R-HSA-6798695,Gene,participates_in,Pathway,MonarchKG::TARKG::iBKH,NCBI_ID,Reactome_ID,alpha-1-B glycoprotein,Neutrophil degranulation,Generalised,Homo species


## 9. QC — NaN Check

In [30]:
def nan_summary(df):
    true_nan   = df.isna().sum()
    string_nan = df.apply(lambda c: c.astype(str).str.upper().eq('NAN').sum())
    return pd.DataFrame({
        'NaN_count':          true_nan,
        "'NAN'_string_count": string_nan,
        'Total_NaN_like':     true_nan + string_nan
    })

nan_summary(final_df_dedup)

,NaN_count,'NAN'_string_count,Total_NaN_like
head,0,0,0
relation,0,0,0
tail,0,0,0
head_type,0,0,0
relation_type,10701,0,10701
tail_type,0,0,0
kg_source,0,0,0
head_id_is,0,0,0
tail_id_is,0,0,0
head_detail_name,0,0,0


In [32]:
set(final_df_dedup['kg_source']), set(final_df_dedup['kg_type'])

({'Harmonizome',
  'Harmonizome::MonarchKG',
  'Harmonizome::MonarchKG::TARKG',
  'Harmonizome::MonarchKG::TARKG::iBKH',
  'Harmonizome::MonarchKG::iBKH',
  'Harmonizome::TARKG',
  'Harmonizome::TARKG::iBKH',
  'Harmonizome::iBKH',
  'MonarchKG',
  'MonarchKG::TARKG',
  'MonarchKG::TARKG::iBKH',
  'MonarchKG::iBKH',
  'TARKG',
  'TARKG::iBKH'},
 {'Generalised'})

In [33]:
final_df_dedup[final_df_dedup['tail_id_is'].isna()]

,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name,kg_type,species


## 10. Save Output

In [35]:

final_df_dedup.to_csv(OUT_PATH, index=False)
print(f"Saved {len(final_df_dedup):,} rows → {OUT_PATH}")

Saved 124,958 rows → /storage/Arushi/090526_EvoAge/kg_formation/processed_data_relation_wise_merge/generalised/GENE_PATHWAY/ALL_GENE_PATHWAY.csv
